In [1]:
# ============================
# 1. Imports
# ============================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB

from sentence_transformers import SentenceTransformer

# ============================
# 2. Load Dataset
# ============================
df = pd.read_csv("merged_logs.csv")

df = df.dropna(subset=["clean_message", "level"])
df = df[["clean_message", "level"]]
df.columns = ["text", "label"]

# ============================
# 3. Encode Labels
# ============================
labels = df["label"].unique()
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}

df["label_id"] = df["label"].map(label2id)

# ============================
# 4. Train-Test Split
# ============================
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["label_id"],
    test_size=0.2,
    random_state=42,
    stratify=df["label_id"]
)

# ============================
# 5. Sentence Transformer
# ============================
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Encoding data...")
X_train_emb = model.encode(X_train.tolist(), batch_size=64, show_progress_bar=True)
X_test_emb = model.encode(X_test.tolist(), batch_size=64, show_progress_bar=True)

# ============================
# 6. Define Models
# ============================
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Linear SVM": LinearSVC(),
    "Naive Bayes": GaussianNB()
}

# ============================
# 7. Train + Evaluate
# ============================
results = {}

for name, clf in models.items():
    print(f"\n===== Training {name} =====")

    clf.fit(X_train_emb, y_train)
    y_pred = clf.predict(X_test_emb)

    acc = accuracy_score(y_test, y_pred)

    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, target_names=id2label.values()))

    results[name] = acc

# ============================
# 8. Compare Results
# ============================
print("\n===== Model Comparison =====")
for model_name, score in results.items():
    print(f"{model_name}: {score:.4f}")

best_model = max(results, key=results.get)
print(f"\nBest Model: {best_model}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding data...


Batches:   0%|          | 0/350 [00:00<?, ?it/s]

Batches:   0%|          | 0/88 [00:00<?, ?it/s]


===== Training Logistic Regression =====
Accuracy: 0.9663
              precision    recall  f1-score   support

        INFO       1.00      0.99      0.99      1400
     WARNING       1.00      1.00      1.00      1400
    CRITICAL       0.89      1.00      0.94      1400
       ERROR       1.00      0.88      0.94      1400

    accuracy                           0.97      5600
   macro avg       0.97      0.97      0.97      5600
weighted avg       0.97      0.97      0.97      5600


===== Training Random Forest =====
Accuracy: 0.9670
              precision    recall  f1-score   support

        INFO       1.00      1.00      1.00      1400
     WARNING       1.00      1.00      1.00      1400
    CRITICAL       0.90      0.98      0.94      1400
       ERROR       0.99      0.89      0.93      1400

    accuracy                           0.97      5600
   macro avg       0.97      0.97      0.97      5600
weighted avg       0.97      0.97      0.97      5600


===== Training Li

In [2]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

In [3]:

le = LabelEncoder()
df["label_id"] = le.fit_transform(df["label"])

id2label = {i: label for i, label in enumerate(le.classes_)}

In [5]:
clf= XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softmax",
        num_class=len(id2label),
        eval_metric="mlogloss",
        use_label_encoder=False
)

In [6]:

print(f"\n===== Training {name} =====")

clf.fit(X_train_emb, y_train)
y_pred = clf.predict(X_test_emb)

acc = accuracy_score(y_test, y_pred)

print(f"Accuracy: {acc:.4f}")
print(classification_report(y_test, y_pred, target_names=id2label.values()))


===== Training Naive Bayes =====


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:00:46] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Accuracy: 0.9666
              precision    recall  f1-score   support

    CRITICAL       1.00      1.00      1.00      1400
       ERROR       1.00      1.00      1.00      1400
        INFO       0.90      0.98      0.94      1400
     WARNING       0.98      0.89      0.93      1400

    accuracy                           0.97      5600
   macro avg       0.97      0.97      0.97      5600
weighted avg       0.97      0.97      0.97      5600

